# SaaSquatch Signal — Interactive API & Scoring Walkthrough

> **"From thousands of companies to the companies worth pursuing first."**

This Jupyter Notebook demonstrates the **SaaSquatch Signal API**, the deterministic 5-factor scoring engine, and pipeline prioritization logic built for acquisition entrepreneurs and M&A deal teams.

## 1. Setup & Environment
We import standard data science and HTTP libraries to interact with the running SaaSquatch Signal API.

In [ ]:
import requests
import pandas as pd
import json

BASE_URL = 'http://localhost:3001/api'

# Check API health and connectivity
try:
    resp = requests.get(f'{BASE_URL}/leads?limit=1')
    if resp.status_code == 200:
        print('✅ Successfully connected to SaaSquatch Signal API!')
    else:
        print(f'API responded with status code: {resp.status_code}')
except Exception as e:
    print(f'Note: Server not detected locally at {BASE_URL}. Offline mock mode available. Error: {e}')

## 2. Inspecting the Active Acquisition Mandate (ICP)
Let's inspect the active Ideal Customer Profile (ICP) governing the fund's investment thesis.

In [ ]:
icp_resp = requests.get(f'{BASE_URL}/icp')
active_icp = icp_resp.json().get('icp', {})

print(f"Mandate Name:      {active_icp.get('name')}")
print(f"Target Revenue:    ${active_icp.get('minRevenue'):,} – ${active_icp.get('maxRevenue'):,}")
print(f"Target Headcount:  {active_icp.get('minEmployees')} – {active_icp.get('maxEmployees')} employees")
print(f"Target Industries: {', '.join(active_icp.get('industries', []))}")
print(f"Target Regions:    {', '.join(active_icp.get('locations', []))}")

## 3. Querying & Prioritizing Leads
Let's fetch the full pipeline of leads scored and ranked by the 5-factor scoring engine.

In [ ]:
leads_resp = requests.get(f'{BASE_URL}/leads?limit=50&sortBy=score&sortOrder=desc')
leads_data = leads_resp.json().get('data', [])

df = pd.DataFrame([
    {
        'Company': l.get('companyName'),
        'Tier': l.get('priority'),
        'Score': l.get('score'),
        'Industry': l.get('industry'),
        'Revenue ($)': l.get('revenue'),
        'Employees': l.get('employees'),
        'Decision Maker': l.get('contact', {}).get('name'),
        'Title': l.get('contact', {}).get('title'),
    }
    for l in leads_data
])

print(f"Total leads evaluated: {len(df)}")
print("\n--- Tier Distribution ---")
print(df['Tier'].value_counts())

# Display Top 10 High-Priority Acquisition Targets
df.head(10)

## 4. Deep-Dive: Explainability & Deal Signals
For any high-priority target (Tier A), the system exposes exact factor breakdowns and structured deal signals rather than a black-box number.

In [ ]:
if leads_data:
    top_lead_id = leads_data[0].get('_id')
    detail_resp = requests.get(f'{BASE_URL}/leads/{top_lead_id}')
    detail = detail_resp.json()
    
    lead = detail.get('lead', {})
    signals = detail.get('dealSignals', [])
    
    print(f"Company: {lead.get('companyName')}")
    print(f"Score:   {lead.get('score')}/100 (Tier {lead.get('priority')})")
    print(f"Key Rationale:")
    for reason in lead.get('scoreReasons', []):
        print(f"  • {reason}")
        
    print("\nStructured Deal Signals:")
    for sig in signals:
        icon = '✅' if sig.get('met') else '❌'
        print(f"  {icon} {sig.get('label')}: {sig.get('detail')}")

## 5. Live ICP Re-Scoring Demonstration
When an acquisition entrepreneur pivots their search mandate (e.g. shifts focus from general SaaS to Cybersecurity with $5M+ revenue), the system re-scores all companies instantly.

In [ ]:
# Update ICP to a targeted Cybersecurity profile
new_icp_payload = {
    'name': 'Enterprise Cybersecurity & Infrastructure Focus',
    'industries': ['Cybersecurity', 'Cloud Infrastructure'],
    'minRevenue': 3000000,
    'maxRevenue': 20000000,
    'minEmployees': 20,
    'maxEmployees': 150,
    'locations': ['United States', 'Canada'],
    'isActive': True
}

save_resp = requests.post(f'{BASE_URL}/icp', json=new_icp_payload)
print(save_resp.json().get('message', 'Updated successfully!'))

# Re-query leads to see the new ranking
updated_leads = requests.get(f'{BASE_URL}/leads?limit=5&sortBy=score&sortOrder=desc').json().get('data', [])
print("\n--- New Top 5 Targets Under Updated Mandate ---")
for idx, l in enumerate(updated_leads, 1):
    print(f"{idx}. {l.get('companyName')} — Score: {l.get('score')} ({l.get('priority')}) | {l.get('industry')}")

## Summary
SaaSquatch Signal bridges raw company discovery to high-conversion targeted outreach by providing:
1. **Configurable ICP Mandates**
2. **Transparent, Multi-Factor Scoring (0–100, Tiers A–D)**
3. **Structured Deal Signals & Actionable Outreach Insights**
4. **Full CSV Data Export & Sales Workflow Integration**